# Stage 13 — Scientific Interpretation

| Field | Value |
|---|---|
| **Pipeline stage** | Stage 13 — Scientific interpretation |
| **Previous stage** | Stage 12 — Mars threshold analysis (`notebooks/mars/05_mars_threshold_sensitivity.ipynb`) |
| **Next stage** | Stage 14 — Figures and reporting (`notebooks/presentation/result_figures.ipynb`) |
| **Purpose** | Translate Mars model outputs into geomorphological meaning. Computes coupling rates by network, shows geographic distribution, compares regimes, and frames the main scientific findings. |
| **Inputs** | `data/Mars/model_outputs/mars_combined_reg{A,B,C}_predictions.parquet`; `data/Mars/model_outputs/mars_combined_reg{A,B,C}_by_network.csv`; `data/Mars/topology/mars_vn_topology_model_ready.gpkg` |
| **Outputs** | Summary tables and figures. Science narrative sections are marked `[FILL IN]` for author completion. |
| **Note** | All computation is complete. Scientific interpretation text is skeleton — replace `[FILL IN]` sections with author analysis. |

## 0. Configuration

In [ ]:
# Operating threshold per regime — use Earth-calibrated values by default (None = read from models/).
THRESHOLD_OVERRIDE: dict[str, float] | None = None  # e.g. {'regA': 0.70, 'regB': 0.72, 'regC': 0.74}

# Minimum pairs per network to include in coupling-rate analysis.
MIN_PAIRS_PER_NETWORK = 3

# High-confidence threshold for the "robust touching" count.
HIGH_CONF_THRESHOLD = 0.85

## 1. Load predictions and thresholds

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from channel_heads.io.paths import MARS_MODEL_OUTPUTS_DIR, MODELS_DIR, MARS_DIR

REGIMES_ORDER = ['regA', 'regB', 'regC']
COLORS = {'regA': '#e41a1c', 'regB': '#377eb8', 'regC': '#4daf4a'}

preds: dict[str, pd.DataFrame] = {}
thresholds: dict[str, float] = {}
by_net: dict[str, pd.DataFrame] = {}

for reg in REGIMES_ORDER:
    pred_path = MARS_MODEL_OUTPUTS_DIR / f"mars_combined_{reg}_predictions.parquet"
    thr_path  = MODELS_DIR / f"optimal_threshold_geom_plus_cnn_emb_{reg}.txt"
    net_path  = MARS_MODEL_OUTPUTS_DIR / f"mars_combined_{reg}_by_network.csv"

    if not pred_path.exists():
        print(f"WARNING: {pred_path} not found — skipping {reg}")
        continue

    preds[reg] = pd.read_parquet(pred_path)

    if THRESHOLD_OVERRIDE and reg in THRESHOLD_OVERRIDE:
        thresholds[reg] = THRESHOLD_OVERRIDE[reg]
    elif thr_path.exists():
        thresholds[reg] = float(thr_path.read_text().strip())
    else:
        thresholds[reg] = 0.5
        print(f"WARNING: No threshold file for {reg}, using 0.5")

    if net_path.exists():
        by_net[reg] = pd.read_csv(net_path)

    n_total = len(preds[reg])
    n_touch = int((preds[reg]['prob_touching'] >= thresholds[reg]).sum())
    print(f"{reg}: thr={thresholds[reg]:.4f}  {n_touch:,}/{n_total:,} pairs predicted touching ({100*n_touch/n_total:.1f}%)")

## 2. Main result — coupling rates by regime

**[FILL IN]** Interpret what these coupling rates mean for Martian geomorphology.

In [ ]:
coupling_rows = []
for reg, df in preds.items():
    thr = thresholds[reg]
    df2 = df.copy()
    df2['pred'] = (df2['prob_touching'] >= thr).astype(int)
    df2['high_conf'] = (df2['prob_touching'] >= HIGH_CONF_THRESHOLD).astype(int)

    # Network-level coupling
    net_grp = df2.groupby('network_id').agg(
        n_pairs=('pred', 'count'),
        n_touching=('pred', 'sum'),
        n_high_conf=('high_conf', 'sum'),
        mean_prob=('prob_touching', 'mean'),
    ).reset_index()
    net_grp = net_grp[net_grp['n_pairs'] >= MIN_PAIRS_PER_NETWORK].copy()
    net_grp['touching_frac'] = net_grp['n_touching'] / net_grp['n_pairs']
    net_grp['regime'] = reg

    coupling_rows.append({
        'regime': reg,
        'threshold': round(thr, 4),
        'n_pairs_total': len(df2),
        'n_touching': int(df2['pred'].sum()),
        'touching_pct': round(100 * df2['pred'].mean(), 2),
        'n_high_conf': int(df2['high_conf'].sum()),
        'high_conf_pct': round(100 * df2['high_conf'].mean(), 2),
        'n_networks_analysed': len(net_grp),
        'median_network_coupling': round(net_grp['touching_frac'].median(), 3),
        'pct_networks_any_touching': round(100 * (net_grp['n_touching'] > 0).mean(), 1),
    })

coupling_df = pd.DataFrame(coupling_rows)
print("=== MAIN RESULT: Mars coupling rates ===")
display(coupling_df)

### Scientific interpretation — coupling rates

> **[FILL IN]**
>
> Key questions to address:
> - Are the coupling rates consistent across regimes? What does consistency (or divergence) imply?
> - How do the Mars coupling rates compare to Earth? (Earth has ~[X]% touching pairs in the training set.)
> - Which networks show the highest coupling, and what do they have in common geologically?

## 3. Network-level coupling distribution

In [ ]:
all_net = []
for reg, df in preds.items():
    thr = thresholds[reg]
    df2 = df.copy()
    df2['pred'] = (df2['prob_touching'] >= thr).astype(int)
    net_grp = df2.groupby('network_id').agg(
        n_pairs=('pred', 'count'),
        n_touching=('pred', 'sum'),
        mean_prob=('prob_touching', 'mean'),
    ).reset_index()
    net_grp = net_grp[net_grp['n_pairs'] >= MIN_PAIRS_PER_NETWORK].copy()
    net_grp['touching_frac'] = net_grp['n_touching'] / net_grp['n_pairs']
    net_grp['regime'] = reg
    all_net.append(net_grp)

all_net_df = pd.concat(all_net, ignore_index=True)

# CDF of network coupling fraction
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (title, metric, xlabel) in zip(axes, [
    ('Network coupling fraction CDF', 'touching_frac', 'Touching fraction per network'),
    ('Network mean P(touching) CDF',  'mean_prob',     'Mean P(touching) per network'),
]):
    for reg in preds:
        sub = all_net_df[all_net_df['regime'] == reg].sort_values(metric)
        cdf = np.linspace(0, 1, len(sub))
        ax.plot(sub[metric].values, cdf, color=COLORS[reg], label=reg, linewidth=2)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Cumulative fraction of networks')
    ax.set_title(title)
    ax.legend()
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))

fig.tight_layout()
plt.show()

## 4. Geographic distribution (requires geopandas + topology)

Map network-level coupling rate to Mars valley positions.

In [ ]:
try:
    import geopandas as gpd
    import rasterio
    from rasterio.enums import Resampling
    HAS_GEO = True
except ImportError as e:
    HAS_GEO = False
    print(f"Geographic packages not available ({e}). Skipping map.")

TOPOLOGY_GPKG = MARS_DIR / 'topology' / 'mars_vn_topology_model_ready.gpkg'
HILLSHADE_PATH = MARS_DIR / 'MOLA_Hillshade_Robinson_128ppd.tif'

if HAS_GEO and TOPOLOGY_GPKG.exists():
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        try:
            edges_gdf = gpd.read_file(TOPOLOGY_GPKG, layer='edges')
        except Exception as exc:
            edges_gdf = None
            print(f"Could not load edges layer: {exc}")

    if edges_gdf is not None and 'network_id' in edges_gdf.columns:
        # Use regA at Earth-calibrated threshold as the primary result
        reg = 'regA'
        if reg not in preds:
            reg = list(preds.keys())[0]

        thr = thresholds[reg]
        df2 = preds[reg].copy()
        df2['pred'] = (df2['prob_touching'] >= thr).astype(int)
        net_coup = df2.groupby('network_id').agg(
            touching_frac=('pred', 'mean'),
            mean_prob=('prob_touching', 'mean'),
            n_pairs=('pred', 'count'),
        ).reset_index()

        edges_merged = edges_gdf.merge(
            net_coup[['network_id', 'touching_frac', 'mean_prob']],
            on='network_id', how='left',
        )

        fig, ax = plt.subplots(figsize=(16, 8))

        # MOLA hillshade background
        if HILLSHADE_PATH.exists():
            with rasterio.open(HILLSHADE_PATH) as src:
                scale = 1024 / max(src.width, src.height)
                out_w = max(1, int(src.width * scale))
                out_h = max(1, int(src.height * scale))
                hs = src.read(1, out_shape=(out_h, out_w),
                              resampling=Resampling.bilinear).astype(float)
                extent = [src.bounds.left, src.bounds.right,
                          src.bounds.bottom, src.bounds.top]
                hs_crs = src.crs
            ax.imshow(hs, cmap='gray', extent=extent, aspect='auto',
                      vmin=np.nanpercentile(hs, 5), vmax=np.nanpercentile(hs, 95))
            edges_reproj = edges_merged.to_crs(hs_crs)
        else:
            edges_reproj = edges_merged

        cmap = plt.cm.RdYlGn
        edges_reproj[edges_reproj['touching_frac'].isna()].plot(
            ax=ax, color='gray', linewidth=0.3, alpha=0.3, label='no data'
        )
        im = edges_reproj[edges_reproj['touching_frac'].notna()].plot(
            ax=ax, column='touching_frac', cmap=cmap,
            vmin=0, vmax=1, linewidth=0.8, alpha=0.9,
            legend=True,
            legend_kwds={'label': 'Touching fraction', 'shrink': 0.5},
        )
        ax.set_title(f'Mars channel-head coupling rate ({reg}, thr={thr:.3f})', fontsize=12)
        fig.tight_layout()
        plt.show()
    else:
        print("Cannot map: edges layer not found or no network_id column.")
else:
    print("geopandas/rasterio not available or topology GeoPackage not found.")

### Scientific interpretation — geographic distribution

> **[FILL IN]**
>
> Key questions to address:
> - Are high-coupling networks clustered in specific terrain types (e.g. ancient highland terrain, volcanic regions, sedimentary basins)?
> - Is there a latitudinal or hemispheric gradient in coupling rate?
> - Do high-coupling networks correspond to networks with high DD (dense Mars valleys), suggesting a geomorphic link?

## 5. Feature analysis — what distinguishes touching pairs on Mars?

In [ ]:
FEATURE_COLS = [
    'delta_L', 'orientation_diff_deg', 'headhead_dist_norm',
    'apex_angle_deg', 'proximity_profile_norm',
]

# Use regA for feature analysis
reg = 'regA' if 'regA' in preds else list(preds.keys())[0]
df = preds[reg].copy()
thr = thresholds[reg]
df['pred'] = (df['prob_touching'] >= thr).astype(int)

available_feats = [f for f in FEATURE_COLS if f in df.columns]

if available_feats:
    fig, axes = plt.subplots(1, len(available_feats), figsize=(14, 3.5))
    if len(available_feats) == 1:
        axes = [axes]

    for ax, feat in zip(axes, available_feats):
        for lbl, color, label in [
            (1, '#d62728', 'pred touching'),
            (0, '#1f77b4', 'pred non-touching'),
        ]:
            vals = df[df['pred'] == lbl][feat].dropna()
            if len(vals):
                lo, hi = vals.quantile(0.01), vals.quantile(0.99)
                ax.hist(vals.clip(lo, hi), bins=35, alpha=0.55,
                        color=color, label=label, density=True)
        ax.set_title(feat, fontsize=8)
        ax.set_yticks([])

    axes[0].legend(fontsize=7)
    fig.suptitle(f'Mars feature distributions by predicted label ({reg})', fontsize=9)
    fig.tight_layout()
    plt.show()
else:
    print(f"Feature columns not found. Available: {df.columns.tolist()}")

### Scientific interpretation — feature signatures of Mars touching pairs

> **[FILL IN]**
>
> Key questions to address:
> - Do Mars touching pairs have similar feature signatures to Earth touching pairs, or do the distributions shift?
> - If the feature distributions are shifted on Mars, what does that imply about the extrapolation assumption?
> - Which features are most discriminative for Mars pairs?

## 6. Cross-regime consistency

In [ ]:
# For pairs that appear in all regimes (same pair_id), compare predictions.
common_ids = None
for reg, df in preds.items():
    ids = set(df['pair_id'].dropna().astype(str))
    common_ids = ids if common_ids is None else common_ids & ids

if common_ids and len(common_ids) > 0:
    print(f"Pairs appearing in all regimes: {len(common_ids):,}")
    sub = {reg: df[df['pair_id'].astype(str).isin(common_ids)].sort_values('pair_id')
           for reg, df in preds.items()}

    # Correlation matrix of probabilities
    prob_df = pd.DataFrame({
        reg: df.set_index('pair_id')['prob_touching']
        for reg, df in sub.items()
    })
    corr = prob_df.corr()
    print("Correlation of P(touching) across regimes:")
    display(corr.round(3))
else:
    print("No common pair_ids across all regimes — pairs are regime-specific (different network extractions).")
    print("Cross-regime comparison is at the network level (see Section 2).")

## 7. Open scientific questions

> **[FILL IN]** Use this section to record open questions and hypotheses for follow-up.
>
> Suggested structure:
>
> 1. **Main finding:** [state the primary quantitative result]
> 2. **Key uncertainty:** [what does the model NOT tell us?]
> 3. **Geologic context:** [which Mars terrain types are most coupled, and why?]
> 4. **Comparison with prior work:** [how do these results relate to Goren & Shelef 2024 on Earth?]
> 5. **Next steps:** [what analysis would strengthen or challenge the main finding?]